[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/notebooks/name-generation-rnn.ipynb)

# Character-Level Name Generation with RNN

Simple implementation of a character-level RNN to generate names. Train on real names and generate new ones.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as L
from torch.utils.data import Dataset, DataLoader
import numpy as np
import urllib.request

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)
L.seed_everything(42)

print(f"PyTorch: {torch.__version__}")
print(f"Lightning: {L.__version__}")

## Data Loading

In [ ]:
# Download and load names dataset
url = 'https://raw.githubusercontent.com/karpathy/makemore/refs/heads/master/names.txt'
urllib.request.urlretrieve(url, 'names.txt')

with open('names.txt', 'r') as f:
    names = f.read().splitlines()

names = [name.strip().lower() for name in names if name.strip()]
print(f"Loaded {len(names)} names")

## Vocabulary

In [ ]:
# Build character vocabulary
chars = sorted(list(set(''.join(names))))
chars = ['.'] + chars  # Add special start/end token

char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}
vocab_size = len(chars)

print(f"Vocabulary: {vocab_size} characters")

## Dataset

In [ ]:
class NamesDataset(Dataset):
    def __init__(self, names, char_to_idx, max_length=None):
        self.names = names
        self.char_to_idx = char_to_idx
        self.max_length = max_length or max(len(n) for n in names) + 1
    
    def __len__(self):
        return len(self.names)
    
    def __getitem__(self, idx):
        name = self.names[idx]
        name_with_tokens = '.' + name + '.'
        indices = [self.char_to_idx[ch] for ch in name_with_tokens]
        
        x = torch.tensor(indices[:-1], dtype=torch.long)
        y = torch.tensor(indices[1:], dtype=torch.long)
        return x, y

# Create and split dataset
dataset = NamesDataset(names, char_to_idx)
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

In [ ]:
def collate_fn(batch):
    xs, ys = zip(*batch)
    xs_padded = nn.utils.rnn.pad_sequence(xs, batch_first=True, padding_value=0)
    ys_padded = nn.utils.rnn.pad_sequence(ys, batch_first=True, padding_value=-100)
    return xs_padded, ys_padded

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, collate_fn=collate_fn)

## Model (Character-level RNN)

In [ ]:
class NameGeneratorRNN(L.LightningModule):
    def __init__(self, vocab_size: int, embedding_dim: int = 64, hidden_size: int = 256,
                 num_layers: int = 2, dropout: float = 0.2, learning_rate: float = 1e-3):
        super().__init__()
        self.save_hyperparameters()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, num_layers, batch_first=True,
                          dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, vocab_size)
        self.criterion = nn.CrossEntropyLoss()
    
    def forward(self, x, hidden=None):
        embedded = self.embedding(x)
        rnn_out, hidden = self.rnn(embedded, hidden)
        rnn_out = self.dropout(rnn_out)
        logits = self.fc(rnn_out)
        return logits, hidden
    
    def training_step(self, batch, batch_idx):
        x, y = batch
        logits, _ = self(x)
        loss = self.criterion(logits.view(-1, self.hparams.vocab_size), y.view(-1))
        self.log('train_loss', loss, prog_bar=True)
        return loss
    
    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits, _ = self(x)
        loss = self.criterion(logits.view(-1, self.hparams.vocab_size), y.view(-1))
        self.log('val_loss', loss, prog_bar=True)
        return loss
    
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)
    
    @torch.no_grad()
    def generate(self, char_to_idx, idx_to_char, max_length=20, temperature=1.0, num_samples=1):
        self.eval()
        generated_names = []
        
        for _ in range(num_samples):
            current_idx = char_to_idx['.']
            name_chars = []
            hidden = None
            
            for _ in range(max_length):
                x = torch.tensor([[current_idx]], dtype=torch.long, device=self.device)
                logits, hidden = self(x, hidden)
                probs = F.softmax(logits[0, -1] / temperature, dim=0)
                next_idx = torch.multinomial(probs, 1).item()
                next_char = idx_to_char[next_idx]
                
                if next_char == '.':
                    break
                
                name_chars.append(next_char)
                current_idx = next_idx
            
            generated_names.append(''.join(name_chars))
        
        return generated_names

model = NameGeneratorRNN(vocab_size=vocab_size)

## Training

In [ ]:
trainer = L.Trainer(max_epochs=30, accelerator='auto', devices=1, 
                    enable_progress_bar=True, log_every_n_steps=20)
trainer.fit(model, train_loader, val_loader)

## Generation

In [ ]:
# Generate sample names
generated = model.generate(char_to_idx, idx_to_char, max_length=15, temperature=0.8, num_samples=20)
generated = [name.capitalize() for name in generated]
print(", ".join(generated))

In [ ]:
# Analyze novelty (new vs existing names)
sample = model.generate(char_to_idx, idx_to_char, max_length=15, temperature=0.8, num_samples=100)
sample = [name for name in sample if name]

original_names_set = set(names)
new_names = [name for name in sample if name not in original_names_set]
existing_names = [name for name in sample if name in original_names_set]

print(f"Total: {len(sample)}, Unique: {len(set(sample))}")
print(f"✨ NEW: {len(new_names)} ({len(new_names)/len(sample)*100:.1f}%)")
print(f"♻️  EXISTING: {len(existing_names)} ({len(existing_names)/len(sample)*100:.1f}%)")

print(f"\nNEW: {', '.join([n.capitalize() for n in new_names[:10]])}")
print(f"EXISTING: {', '.join([n.capitalize() for n in existing_names[:10]])}")